In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score,confusion_matrix
import joblib

In [2]:
df=pd.read_csv('student_learning.csv')
print("Top 5 rows: ",df.head(5))
print("Column Datatype: ",df.dtypes)
print("Counting misiing values: ",df.isna().sum())

Top 5 rows:     hours_studied  attendance  practice_tests  sleep_hours study_method  \
0            7.5          54               0          5.6        video   
1            4.4          61               0          6.0        video   
2            8.2          63               1          5.2        group   
3            6.8          64               1          8.4         self   
4            1.3          79               5          5.8         self   

   final_score  passed  
0           48       0  
1           50       0  
2           73       1  
3           53       0  
4           51       0  
Column Datatype:  hours_studied     float64
attendance          int64
practice_tests      int64
sleep_hours       float64
study_method       object
final_score         int64
passed              int64
dtype: object
Counting misiing values:  hours_studied     0
attendance        0
practice_tests    0
sleep_hours       0
study_method      0
final_score       0
passed            0
dtype: int64

In [3]:
X=df.drop(columns=["final_score","passed"])
y=df["passed"]
#splitting Training and Testing Data
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

# Seperating numerical and categorical columns
num_features=['attendance','hours_studied','practice_tests','sleep_hours']
cat_features=['study_method']

#filling missing values with the median
numerical_transformer=Pipeline([('imputer',SimpleImputer(strategy="median")),('scaler',StandardScaler())])
# filling missing values with the most frequent and Converting Categorical data into int using one-hot-encoder
categorical_transformer=Pipeline([('imputer',SimpleImputer(strategy="most_frequent")),('enc',OneHotEncoder(handle_unknown='ignore'))])


In [4]:
# Creating Baseline
dummy=DummyClassifier(strategy='most_frequent')
dummy.fit(X_train,Y_train)

DummyClassifier(strategy='most_frequent')

In [5]:
prep=ColumnTransformer([('num',numerical_transformer,num_features),('category',categorical_transformer,cat_features)])

#training logistic regression model
logistic_pipeline=Pipeline([("Pre-processing",prep),('model',LogisticRegression(max_iter=500))])
logistic_pipeline.fit(X_train,Y_train)

rf_pipeline=Pipeline([("Pre-processing",prep),('model',RandomForestClassifier(max_depth=4,n_estimators=500,random_state=42))])
rf_pipeline.fit(X_train,Y_train)

Pipeline(steps=[('Pre-processing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['attendance',
                                                   'hours_studied',
                                                   'practice_tests',
                                                   'sleep_hours']),
                                                 ('category',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('enc',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['study_method'])])),
                ('model',
                 RandomForestClassifier(max_depth=4, n_estimators=500,
                                        random_state=42))])

In [6]:
# Accuracy, f1-score and confusion matrix of all models
model={ "Dummy":dummy,
        "logistic regression":logistic_pipeline,
        "random forest":rf_pipeline}
for name,model in model.items():
    y_pred=model.predict(X_test)
    print(f"{'_'*5}{ name}{'_'*5}")
    print(f"Accuracy: {accuracy_score(Y_test,y_pred)}")
    print(f"F1-score: {f1_score(Y_test,y_pred)}")
    print(f"Confusion matrix: {confusion_matrix(Y_test,y_pred)}")


_____Dummy_____
Accuracy: 0.5535714285714286
F1-score: 0.0
Confusion matrix: [[31  0]
 [25  0]]
_____logistic regression_____
Accuracy: 0.75
F1-score: 0.72
Confusion matrix: [[24  7]
 [ 7 18]]
_____random forest_____
Accuracy: 0.7857142857142857
F1-score: 0.75
Confusion matrix: [[26  5]
 [ 7 18]]


**Baseline:**
Accuracy: 0.5535714285714286
F1-score: 0.0
Confusion matrix: [[31  0]
 [25  0]]

**Logistic regression**
Accuracy: 0.75
F1-score: 0.72
Confusion matrix: [[24  7]
 [ 7 18]]

**Random forest**
Accuracy: 0.7857142857142857
F1-score: 0.75
Confusion matrix: [[26  5]
 [ 7 18]]"""

Dummy classifier showed accuracy of 0.55 where it's TN are 31 FP are 0 FN are 25 and TN are 0. As the strategy was most frequent it predicted the 0 class for everything.

Logistic regression showed an accuracy of 75% with TN 24, FP 7, FN 7 and TP 18. It predicted 24 actual class 0 and  18 actual class 1.

Random Forest showed an accuracy of 78% with TN 26, FP 5, FN 7 and TP 18. It predicted 26 actual class 0 and  18 actual class 1.

**Conclusion:**
 Random Forest Performs the best so the final chosen model is Random Forest


In [7]:
model_dump=joblib.dump(rf_pipeline,"model.joblib")
print("Model dumped")

Model dumped


In [8]:
# Loaded the model
model = joblib.load('model.joblib')
# Retreiving the newly created new_student_records csv
new_data = pd.read_csv('new_student_records.csv')

# passing data for predictions
predictions = model.predict(new_data)

# Creating new column named passed storing the predictions
new_data['passed'] = predictions

# Display the predictions
print("\nPredictions for New Students:")
print(new_data[['hours_studied', 'attendance', 'study_method','passed']])



Predictions for New Students:
    hours_studied  attendance study_method  passed
0             3.8          70        video       0
1             1.4          58         self       0
2             8.0          93         self       1
3             5.8          92         self       1
4             1.5          54        video       0
5             7.2          85        group       1
6             2.1          62         self       0
7             9.5          98        group       1
8             4.0          75        video       0
9             6.5          88         self       1
10            3.0          65        video       0
11            8.8          95        group       1
